# Анализ NL2SQL-бенчмарка\nNotebook-first сценарий с автоматической загрузкой `.env`.

In [ ]:
from pathlib import Path\nimport sys\nimport os\nimport json\nimport pandas as pd

In [ ]:
PROJECT_ROOT = Path.cwd().resolve().parent\nif str(PROJECT_ROOT) not in sys.path:\n    sys.path.insert(0, str(PROJECT_ROOT))\n\nfrom src.utils.env_loader import load_dotenv_file\nfrom src.inference.runner import run_experiment\nfrom src.inference.batch_runner import run_batch

In [ ]:
# Загружаем .env один раз за сессию ноутбука.\nload_dotenv_file(PROJECT_ROOT / '.env')\n{k: os.getenv(k) for k in ['L2SB_DATASET', 'L2SB_DATASET_PATH', 'L2SB_MODEL_KEY', 'L2SB_OUTPUT_DIR']}

In [ ]:
# Раннер также автоматически подхватывает .env.\nresult = run_experiment({})\nresult

In [ ]:
# Пакетный запуск из конфига; список можно задать через L2SB_EXPERIMENT_NAMES\nexperiments_config = os.getenv('L2SB_EXPERIMENTS_CONFIG', str(PROJECT_ROOT / 'configs' / 'experiments.yaml'))\n# результаты_batch = run_batch(experiments_config)\n# результаты_batch

In [ ]:
run_files = sorted((PROJECT_ROOT / 'results' / 'runs').glob('*.json'))\nrows = [json.loads(path.read_text(encoding='utf-8')) for path in run_files]\ndf = pd.DataFrame(rows)\ndf[['model', 'model_name', 'dataset', 'execution_accuracy', 'pass_at_k', 'avg_latency']].sort_values(['dataset', 'model'])

In [ ]:
if not df.empty:\n    summary = (\n        df.groupby(['dataset', 'model_name'], as_index=False)[['execution_accuracy', 'pass_at_k', 'avg_latency']]\n        .mean()\n    )\n    display(summary)\n    summary.plot.bar(x='model_name', y=['execution_accuracy', 'pass_at_k'], figsize=(9, 4), title='Метрики NL2SQL по моделям')